<a href="https://colab.research.google.com/github/AngelDioses/Miner-a-de-Datos_Tarea-6/blob/main/Practica6_Dioses%20Angel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica 6 — Ingeniería de Características y Reducción de Dimensionalidad
**UNMSM – FISI · Minería de Datos · Dr. José Herrera · 2026-1**

**Dataset:** FIFA 20 Complete Player Dataset (Stefano Leone, Kaggle) + dataset simulado de lesiones.

**Contenido**
- Caso 1 · Construcción de Características
- Caso 2 · Integración y Formateo
- Caso 3 · Colinealidad e Importancia
- Caso 4 · PCA vs LDA

> Apellidos y Nombres: _________________________


## 0 · Configuración y carga del dataset

Se intenta primero la descarga real desde el espejo de GitHub (Opción A). Si no hay
internet, se cae automáticamente al **dataset simulado** (Opción C) para que todos los
enunciados funcionen igual.

In [ ]:
# Librerías base
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
RANDOM_STATE = 42


In [ ]:
# ---- Carga del dataset FIFA ----
# OPCIÓN A — Descarga directa desde GitHub (sin cuenta Kaggle)
URL = ("https://raw.githubusercontent.com/apoorva-21/"
       "fifa-analysis/master/data/players_20.csv")

cols_utiles = ["short_name", "age", "height_cm", "weight_kg", "overall",
               "potential", "value_eur", "wage_eur", "player_positions",
               "preferred_foot", "international_reputation", "skill_moves",
               "weak_foot", "pace", "shooting", "passing", "dribbling",
               "defending", "physic"]

try:
    df = pd.read_csv(URL)
    # Nos quedamos solo con las columnas que pide la práctica (el CSV real tiene >100)
    df = df[[c for c in cols_utiles if c in df.columns]].copy()
    print("Dataset REAL cargado desde GitHub:", df.shape)
except Exception as e:
    print("Sin internet / fallo de URL -> uso dataset simulado (Opción C). Detalle:", e)
    # OPCIÓN C — dataset simulado reproducible
    np.random.seed(42); n = 500
    df = pd.DataFrame({
        "short_name": [f"Jugador_{i}" for i in range(n)],
        "age":        np.random.randint(17, 38, n),
        "height_cm":  np.random.normal(181, 7, n).round().astype(int),
        "weight_kg":  np.random.normal(75, 8, n).round().astype(int),
        "overall":    np.random.randint(55, 95, n),
        "potential":  np.random.randint(60, 95, n),
        "value_eur":  np.random.lognormal(15.5, 1.5, n).round(),
        "wage_eur":   np.random.lognormal(9.5, 1.0, n).round(),
        "player_positions": np.random.choice(["ST","CM","CB","GK","RW","LB"], n),
        "preferred_foot":   np.random.choice(["Left","Right"], n, p=[.25,.75]),
        "international_reputation": np.random.choice([1,2,3,4,5], n, p=[.7,.18,.08,.03,.01]),
        "skill_moves": np.random.choice([1,2,3,4,5], n),
        "weak_foot":   np.random.choice([1,2,3,4,5], n),
        "pace":      np.random.normal(70, 12, n).clip(20, 99).round(),
        "shooting":  np.random.normal(60, 15, n).clip(20, 99).round(),
        "passing":   np.random.normal(65, 12, n).clip(20, 99).round(),
        "dribbling": np.random.normal(68, 12, n).clip(20, 99).round(),
        "defending": np.random.normal(55, 18, n).clip(20, 99).round(),
        "physic":    np.random.normal(67, 11, n).clip(20, 99).round(),
    })
    print("Dataset SIMULADO creado:", df.shape)

df.head()


In [ ]:
# Los porteros (GK) tienen NaN en pace/shooting/... en el dataset real.
# Para los ejercicios técnicos los imputamos con la mediana de la columna.
tecnicas = ["pace", "shooting", "passing", "dribbling", "defending", "physic"]
print("Valores faltantes ANTES de imputar:")
print(df[tecnicas + ["value_eur", "wage_eur"]].isnull().sum())

for c in tecnicas:
    df[c] = pd.to_numeric(df[c], errors="coerce")
    df[c] = df[c].fillna(df[c].median())

df = df.dropna(subset=["value_eur", "wage_eur"]).reset_index(drop=True)
print("\nValores faltantes DESPUÉS de imputar:")
print(df[tecnicas].isnull().sum())


---
## CASO 1 · Construcción de Características  (5 pts) — *"El Agente Madrugador"*

Creamos ratios y métricas derivadas que el club usará como filtros automáticos.

### a) Ratio de crecimiento — los "diamantes en bruto"
`growth = potential − overall`. Cuanto mayor, más margen de mejora tiene el jugador.

In [ ]:
df["growth"] = df["potential"] - df["overall"]

diamantes = df.nlargest(5, "growth")[["short_name", "age", "overall", "potential", "growth"]]
print("Top 5 con mayor potencial de crecimiento:")
diamantes


### b) Eficiencia salarial
`value_per_wage = value_eur / (wage_eur + 1)`.

**Interpretación:**
- **Valor alto** → el jugador vale mucho en el mercado pero cobra relativamente poco: está **infravalorado** (gran oportunidad para el agente: comprar barato un activo caro).
- **Valor bajo** → cobra mucho para lo que vale en traspaso: jugador **caro de mantener** / sobrevalorado salarialmente.

In [ ]:
df["value_per_wage"] = df["value_eur"] / (df["wage_eur"] + 1)

infravalorados = df.nlargest(5, "value_per_wage")[
    ["short_name", "value_eur", "wage_eur", "value_per_wage"]]
print("Top 5 más INFRAVALORADOS respecto a su salario:")
infravalorados


### c) IMC y categoría física
`imc = weight_kg / (height_cm/100)**2` y se discretiza en 4 categorías.

In [ ]:
df["imc"] = df["weight_kg"] / (df["height_cm"] / 100) ** 2

df["categoria_fisica"] = pd.cut(
    df["imc"],
    bins=[0, 20, 25, 30, 99],
    labels=["Ligero", "Atlético", "Robusto", "Pesado"])

print(df["categoria_fisica"].value_counts())
df[["short_name", "height_cm", "weight_kg", "imc", "categoria_fisica"]].head()


### d) Índice ofensivo compuesto y etapa de carrera
`idx_off = 0.4*shooting + 0.3*dribbling + 0.3*pace`. Discretizamos la edad.

In [ ]:
df["idx_off"] = 0.4 * df["shooting"] + 0.3 * df["dribbling"] + 0.3 * df["pace"]

df["etapa_carrera"] = pd.cut(
    df["age"],
    bins=[0, 23, 28, 33, 99],
    labels=["Joven", "Óptimo", "Maduro", "Veterano"])

print(df["etapa_carrera"].value_counts())
df[["short_name", "age", "etapa_carrera", "shooting", "dribbling", "pace", "idx_off"]].head()


### e) Encoding categórico (one-hot)
Aplicamos `pd.get_dummies` a `preferred_foot` y `player_positions` con `drop_first=True`.

In [ ]:
cols_antes = df.shape[1]
print("Columnas ANTES del encoding:", cols_antes)

df_encoded = pd.get_dummies(
    df, columns=["preferred_foot", "player_positions"],
    drop_first=True)

cols_despues = df_encoded.shape[1]
print("Columnas DESPUÉS del encoding:", cols_despues)
print("Columnas nuevas creadas:", cols_despues - cols_antes)

# Mostramos algunas dummies generadas
nuevas = [c for c in df_encoded.columns
          if c.startswith("preferred_foot_") or c.startswith("player_positions_")]
df_encoded[nuevas].head()


**Interpretación Caso 1.** A partir de datos crudos generamos métricas que un *scout*
buscaría manualmente: `growth` detecta jóvenes promesas, `value_per_wage` localiza gangas,
`imc`+`categoria_fisica` perfilan el tipo físico, `idx_off` resume la capacidad ofensiva en
un solo número, y el one-hot deja las variables categóricas listas para los modelos.

---
## CASO 2 · Integración y Formateo  (5 pts) — *"Cuidado con las Lesiones"*

In [ ]:
# Dataset 2: lesiones por jugador (simulado, reproducible)
np.random.seed(7)
n_les = int(len(df) * 0.85)
lesiones = pd.DataFrame({
    "id_jugador":         df["short_name"].sample(frac=0.85, random_state=7).values,
    "lesiones_temporada": np.random.poisson(1.2, n_les),
    "dias_baja":          np.random.exponential(20, n_les).round(),
    "tipo_lesion":        np.random.choice(
        ["Muscular", "Articular", "Ósea", "Ninguna"],
        n_les, p=[0.45, 0.30, 0.10, 0.15]),
})
print(lesiones.head())
print("\nFilas de lesiones:", len(lesiones))


### a) Integración (merge left) e imputación

In [ ]:
dfm = pd.merge(df, lesiones,
               left_on="short_name", right_on="id_jugador", how="left")

sin_datos = dfm["id_jugador"].isnull().sum()
print("Jugadores SIN datos médicos:", sin_datos)

# Imputación: 0 lesiones y 0 días de baja donde no hay registro
dfm["lesiones_temporada"] = dfm["lesiones_temporada"].fillna(0)
dfm["dias_baja"] = dfm["dias_baja"].fillna(0)
dfm["tipo_lesion"] = dfm["tipo_lesion"].fillna("Ninguna")

print("NaN tras imputar:",
      dfm[["lesiones_temporada", "dias_baja"]].isnull().sum().to_dict())


### b) Resolución de claves duplicadas

In [ ]:
# id_jugador es redundante (igual a short_name). La eliminamos.
dfm = dfm.drop(columns=["id_jugador"])
print("Filas duplicadas:", dfm.duplicated().sum())
print("Shape tras limpieza:", dfm.shape)


### c) Z-score (StandardScaler) a variables técnicas

In [ ]:
from sklearn.preprocessing import StandardScaler

tecnicas = ["pace", "shooting", "passing", "dribbling", "defending", "physic"]
scaler_z = StandardScaler()
dfm[[c + "_z" for c in tecnicas]] = scaler_z.fit_transform(dfm[tecnicas])

resumen = dfm[[c + "_z" for c in tecnicas]].agg(["mean", "std"]).T
print("Verificación (media ≈ 0, std ≈ 1):")
resumen.round(4)


### d) Min-Max sobre value_eur y wage_eur (con log1p previo)
Estas variables están **muy sesgadas a la derecha**; `np.log1p` comprime la cola y deja una
distribución mucho más simétrica antes de escalar a [0,1].

In [ ]:
from sklearn.preprocessing import MinMaxScaler

dfm["value_log"] = np.log1p(dfm["value_eur"])
dfm["wage_log"]  = np.log1p(dfm["wage_eur"])

scaler_mm = MinMaxScaler()
dfm[["value_mm", "wage_mm"]] = scaler_mm.fit_transform(dfm[["value_log", "wage_log"]])

fig, ax = plt.subplots(2, 2, figsize=(11, 7))
dfm["value_eur"].plot.hist(bins=40, ax=ax[0,0], title="value_eur (original, sesgada)")
dfm["value_log"].plot.hist(bins=40, ax=ax[0,1], title="value_eur tras log1p")
dfm["wage_eur"].plot.hist(bins=40, ax=ax[1,0], title="wage_eur (original, sesgada)")
dfm["wage_log"].plot.hist(bins=40, ax=ax[1,1], title="wage_eur tras log1p")
plt.tight_layout(); plt.show()

print("Rango value_mm:", dfm["value_mm"].min(), "-", dfm["value_mm"].max())


**Comentario (d).** Antes del log, casi todos los jugadores se aglomeran en los valores
bajos y unos pocos cracks forman una cola larguísima. Tras `log1p` el histograma se vuelve
aproximadamente acampanado, lo que evita que esos *outliers* dominen el escalado Min-Max.

### e) One-Hot vs Ordinal — y por qué cada uno
- `tipo_lesion` es **nominal** (no hay orden entre Muscular/Articular/Ósea/Ninguna) → **One-Hot**.
- `etapa_carrera` es **ordinal** (Joven < Óptimo < Maduro < Veterano) → **OrdinalEncoder**, que
  preserva ese orden con un único número creciente.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# One-Hot para la variable nominal
dfm = pd.get_dummies(dfm, columns=["tipo_lesion"], prefix="lesion", drop_first=True)

# Ordinal para la variable con orden natural
orden = [["Joven", "Óptimo", "Maduro", "Veterano"]]
ord_enc = OrdinalEncoder(categories=orden)
dfm["etapa_carrera_ord"] = ord_enc.fit_transform(dfm[["etapa_carrera"]])

print("Dummies de tipo_lesion:",
      [c for c in dfm.columns if c.startswith("lesion_")])
dfm[["etapa_carrera", "etapa_carrera_ord"]].drop_duplicates().sort_values("etapa_carrera_ord")


---
## CASO 3 · Colinealidad e Importancia  (5 pts) — *"Cazando Variables Gemelas"*

### a) Matriz de correlación (heatmap)

In [ ]:
vars_num = ["age", "height_cm", "weight_kg", "overall", "potential",
            "pace", "shooting", "passing", "dribbling", "defending"]

corr = df[vars_num].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", square=True)
plt.title("Matriz de correlación")
plt.tight_layout(); plt.show()

# Pares con |corr| > 0.85 (excluyendo la diagonal)
altos = (corr.where(~np.eye(len(corr), dtype=bool))
             .abs().stack().sort_values(ascending=False))
altos = altos[altos > 0.85].drop_duplicates()
print("Pares con |corr| > 0.85:")
print(altos.head(10))


### b) Cálculo de VIF

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

def tabla_vif(data, cols):
    X = add_constant(data[cols].astype(float))
    vif = pd.DataFrame({
        "Feature": X.columns,
        "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    })
    vif = vif[vif["Feature"] != "const"]          # quitamos la constante
    return vif.sort_values("VIF", ascending=False).reset_index(drop=True)

vif0 = tabla_vif(df, vars_num)
print("VIF inicial:")
vif0


### c) Eliminación iterativa (mientras VIF > 10)

In [ ]:
cols = vars_num.copy()
iteraciones = 0
while True:
    vif = tabla_vif(df, cols)
    peor = vif.iloc[0]
    if peor["VIF"] > 10 and len(cols) > 2:
        cols.remove(peor["Feature"])
        iteraciones += 1
        print(f"Iter {iteraciones}: elimino '{peor['Feature']}' (VIF={peor['VIF']:.1f})")
    else:
        break

print(f"\nIteraciones necesarias: {iteraciones}")
print("Variables finales (VIF aceptable):", cols)
features_finales = cols
tabla_vif(df, features_finales)


### d) Importancia con Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

X = df[features_finales]
y = np.log1p(df["value_eur"])

rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X, y)

imp = (pd.Series(rf.feature_importances_, index=features_finales)
         .sort_values(ascending=False))
top10 = imp.head(10)

plt.figure(figsize=(8, 5))
top10[::-1].plot.barh(color="steelblue")
plt.title("Top-10 importancia de características (Random Forest)")
plt.xlabel("feature_importances_")
plt.tight_layout(); plt.show()
top10


### e) SHAP — comparación con feature_importances_

In [ ]:
try:
    import shap
    muestra = X.sample(min(200, len(X)), random_state=42)
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(muestra)

    shap.summary_plot(shap_values, muestra, plot_type="bar", show=True)

    ranking_shap = (pd.Series(np.abs(shap_values).mean(0), index=muestra.columns)
                      .sort_values(ascending=False))
    comparacion = pd.DataFrame({
        "rank_RF":  imp.rank(ascending=False).astype(int),
        "rank_SHAP": ranking_shap.rank(ascending=False).astype(int)
    }).sort_values("rank_SHAP")
    print(comparacion)
except Exception as e:
    print("SHAP no disponible en este entorno:", e)
    print("En Colab basta con: !pip install shap")


**Comentario (e).** `feature_importances_` mide cuánto reduce el error cada *split* y tiende
a inflar variables con muchos valores distintos. SHAP reparte la contribución de forma más
justa entre features, por lo que es más confiable para el modelo final. Normalmente `overall`
domina ambos rankings, pero el orden de las variables secundarias suele diferir entre métodos.

---
## CASO 4 · PCA vs LDA — Reducción de Dimensionalidad  (5 pts) — *"Comprime y Visualiza"*

### a) Preparación: X escalada y etiqueta posicion_simple

In [ ]:
from sklearn.preprocessing import StandardScaler

# Mapeo posición detallada -> simple {DEL, MED, DEF, POR}
mapa_pos = {
    "ST":"DEL","CF":"DEL","RW":"DEL","LW":"DEL",
    "CAM":"MED","CM":"MED","CDM":"MED","RM":"MED","LM":"MED",
    "CB":"DEF","RB":"DEF","LB":"DEF","RWB":"DEF","LWB":"DEF",
    "GK":"POR",
}
df["pos_primaria"] = df["player_positions"].str.split(",").str[0].str.strip()
df["posicion_simple"] = df["pos_primaria"].map(mapa_pos).fillna("MED")

tecnicas = ["pace", "shooting", "passing", "dribbling", "defending", "physic"]
X = StandardScaler().fit_transform(df[tecnicas])
yv = df["posicion_simple"].values

print("Balance de clases:")
print(df["posicion_simple"].value_counts())


### b) PCA con todos los componentes — Scree Plot

In [ ]:
from sklearn.decomposition import PCA

pca_full = PCA().fit(X)
var_acum = np.cumsum(pca_full.explained_variance_ratio_)
K = int(np.argmax(var_acum >= 0.90) + 1)

plt.figure(figsize=(8,5))
plt.plot(range(1, len(var_acum)+1), var_acum, marker="o")
plt.axhline(0.90, color="red", ls="--", label="90 %")
plt.axvline(K, color="green", ls="--", label=f"K = {K}")
plt.xlabel("Nº de componentes"); plt.ylabel("Varianza explicada acumulada")
plt.title("Scree Plot"); plt.legend(); plt.tight_layout(); plt.show()

print(f"Componentes para >= 90% de varianza: K = {K}")


### c) PCA reducido + loadings

In [ ]:
pca = PCA(n_components=K)
X_pca = pca.fit_transform(X)

loadings = pd.DataFrame(pca.components_.T[:, :2],
                        index=tecnicas, columns=["PC1", "PC2"])
print("Loadings de PC1 y PC2:")
print(loadings.round(3))

print("\nVariable que más aporta a PC1:", loadings["PC1"].abs().idxmax())
print("Variable que más aporta a PC2:", loadings["PC2"].abs().idxmax())


### d) LDA y comparación visual PCA vs LDA

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

lda = LinearDiscriminantAnalysis(n_components=2)
X_lda = lda.fit_transform(X, yv)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for clase in np.unique(yv):
    m = yv == clase
    ax[0].scatter(X_pca[m,0], X_pca[m,1], s=12, alpha=.5, label=clase)
    ax[1].scatter(X_lda[m,0], X_lda[m,1], s=12, alpha=.5, label=clase)
ax[0].set_title("PCA (no supervisado)"); ax[0].set_xlabel("PC1"); ax[0].set_ylabel("PC2")
ax[1].set_title("LDA (supervisado)");    ax[1].set_xlabel("LD1"); ax[1].set_ylabel("LD2")
ax[0].legend(); ax[1].legend()
plt.tight_layout(); plt.show()


**Comentario (d).** El **LDA separa mejor** las posiciones porque usa la etiqueta `y` para
maximizar la distancia entre clases; el PCA solo busca la dirección de máxima varianza sin
conocer las clases, por lo que los grupos se solapan más.

### e) Comparativa de clasificadores (CV = 5)

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

modelos = {
    "LDA": LinearDiscriminantAnalysis(),
    "QDA": QuadraticDiscriminantAnalysis(),
    "LogReg": LogisticRegression(max_iter=1000),
    "SVC(rbf)": SVC(kernel="rbf"),
}

filas = []
for nombre, modelo in modelos.items():
    sc = cross_val_score(modelo, X, yv, cv=5, scoring="accuracy")
    filas.append({"Modelo": nombre, "Accuracy_media": sc.mean(), "Std": sc.std()})

tabla = pd.DataFrame(filas).sort_values("Accuracy_media", ascending=False).reset_index(drop=True)
print(tabla.round(4))


**Comentario (e).** Se compara la media ± desviación de cada clasificador.
El **desempate** ante medias parecidas se decide por la **menor desviación estándar** (modelo
más estable entre folds) y por su simplicidad/interpretabilidad. Típicamente SVC(rbf) o
LogReg logran la mejor exactitud, mientras que LDA es el más simple y robusto.

---
### Conclusiones generales
1. **Feature engineering** convirtió columnas crudas en métricas accionables (`growth`, `value_per_wage`, `idx_off`).
2. La **integración** con lesiones exigió merge, imputación y resolución de claves duplicadas.
3. El **escalado** (Z-score / Min-Max con log) homogeneizó variables de escalas muy distintas (€, kg, cm).
4. **VIF + RF + SHAP** permitieron eliminar variables gemelas y quedarnos con las que explican el valor de mercado.
5. **LDA > PCA** para separar posiciones porque aprovecha la etiqueta de clase.

*— Fin de la Práctica 6 —*